# Basic Maps

Author : Henry Nachman

Date : 10 September 2026

---

This notebook will step through how to create a 'simulated' basic CMB maps with a single point source in Healpy.

In [3]:
import healpy as hp
import os
import numpy as np
from astropy import units as u
import matplotlib.pyplot as plt
import matplotlib

# Define the output directory for the maps
this_dir = os.path.dirname(os.path.abspath("blank_map.ipynb"))
out_dir = os.path.join(this_dir, "maps", "blank_maps")
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

# Matplotlib settings
# matplotlib.style.use('/shared_home/henachman/.matplotlib/stylelib/HNCustomMono.mplstyle')

lat_fwhm = 2*u.arcmin
# Define the coordinates of the point source in sky coordinates
lon = 290.0 * u.deg
lat = -60.0 * u.deg
source_size = (10*u.arcmin).to(u.rad) # point sources for lat are around 1-12 arcmin

# High res refers to nside = 2**11
# Low res refers to nside = 2**9
nside = 2**11
if nside < 2**10:
    res = "lr"
if nside >= 2**11:
    res = "hr"

# Blank Maps

In [ ]:
npix = hp.nside2npix(nside)
resolution = hp.nside2resol(nside, arcmin=True)
print(f"NPIX: {npix}, Resolution: {resolution} arcmin")
empty_map = np.zeros((4, npix))

vec = hp.ang2vec(lon.value, lat.value, lonlat=True)
# Convert the sky coordinates to pixel indices in the HEALPix map
# Find the indices of all the pixels within x degrees of that point

pixel_inds = hp.query_disc(nside, vec=vec, radius=source_size.value, nest=True)
print(len(pixel_inds), source_size, pixel_inds.size//2)
# Add a point source
if len(pixel_inds)==0:
    raise ValueError("Source is too small for pixel resolution. Please increase resolution or sources size.")
empty_map[:, pixel_inds] += 1  # Add a point source with an amplitude of 1 microK

hp.gnomview(empty_map[0], rot=[lon.value, lat.value], xsize=200, nest=True)

output_file_with_source = os.path.join(out_dir, f"{res}_blank_map_{int((source_size.to(u.arcmin)).value)}arcmin.fits")
hp.write_map(output_file_with_source, empty_map, nest=True, overwrite=True)

# With some Cls

In [ ]:
out_dir = os.path.join(this_dir, "maps", "sim_maps")

In [ ]:
npix = hp.nside2npix(nside)
resolution = hp.nside2resol(nside, arcmin=True)
print(f"NPIX: {npix}, Resolution: {resolution} arcmin")

# We will create a blank map
# create a Cl array with zeros
cl = np.zeros((4, hp.sphtfunc.Alm.getsize(nside)))

# The current Cls are 0 - but you can adjust
cl[:, 15] = 10
cl[:, 25] = 5
cl[:, 35] = 5

# Create a blank map using the Cl array
map_ring = hp.synfast(cl, nside=nside, fwhm=lat_fwhm.to_value(u.radian))
output_map = 3e-3 * hp.reorder(map_ring, inp='RING', out='NEST')  # Convert to NESTED ordering and scale to microK  
# Save the blank map to a FITS file
hp.mollview(output_map[0], title="Blank Map", unit="microK", nest=True)

### Point Source

Now we want to add a point source right in the middle of our map

In [ ]:

# Find the vector that points to that point
vec = hp.ang2vec(lon.value, lat.value, lonlat=True)

# Convert the sky coordinates to pixel indices in the HEALPix map
# Find the indices of all the pixels within x degrees of that point
pixel_inds = hp.query_disc(nside, vec=vec, radius=source_size.value, nest=True)
# Add a point source to the map at the specified pixel index
print(pixel_inds.size//2)
output_map[0, pixel_inds] += 6e-2  # Add a point source with an amplitude of 1 microK
# Save the updated map with the point source to a new FITS file
hp.mollview(output_map[0], title="Map with Point Source", unit="microK", nest=True, coord=["C"])

Use Gnomview to zoom in on that spot

In [ ]:
output_file_with_source = os.path.join(out_dir, f"{res}_simple_map_{int((source_size.to(u.arcmin)).value)}arcmin.fits")
hp.write_map(output_file_with_source, output_map, nest=True, overwrite=True)

In [ ]:
# Convert the vector in theta phi to lon lat
hp.gnomview(output_map[0], rot=[lon.value, lat.value], unit="microK", xsize=200, nest=True)
hp.graticule()

In [ ]:
loaded_map = hp.read_map(output_file_with_source, nest=True)
hp.gnomview(loaded_map, rot=[lon.value, lat.value], xsize=2000, nest=True)

# Realistic Cl Map

In [ ]:
ell, DlTT = np.loadtxt("CAMB_fiducial_cosmo_scalCls.dat", usecols=(0, 1), unpack=True)
print(max(ell))
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(ell[2:], DlTT[2:], color="C0")
ax.set_xlim(-100, 5000)
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$D_\ell^{TT}$ [$\mu K^2$]")
ax.set_title("Fiducial CMB Temperature Power Spectrum")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
npix = hp.nside2npix(nside)
resolution = hp.nside2resol(nside, arcmin=True)
print(f"NPIX: {npix}, Resolution: {resolution} arcmin")

cls_input = np.loadtxt("CAMB_fiducial_cosmo_scalCls.dat", usecols=(0,1,2,3), unpack=True)
print(np.shape(cls_input))
# # Create a blank map using the Cl array
map_ring = hp.synfast(cls_input, nside=nside)
output_map = 3e-3 * hp.reorder(map_ring, inp='RING', out='NEST')  # Convert to NESTED ordering and scale to microK  
# Save the blank map to a FITS file
# hp.write_map(output_file, output_map, nest=True, overwrite=True)

hp.mollview(output_map[0], title="Blank Map", unit="microK", nest=True)

In [ ]:
# Find the vector that points to that point
vec = hp.ang2vec(lon.value, lat.value, lonlat=True)

# Convert the sky coordinates to pixel indices in the HEALPix map
# Find the indices of all the pixels within x degrees of that point
pixel_inds = hp.query_disc(nside, vec=vec, radius=source_size.value, nest=True)
# Add a point source to the map at the specified pixel index
output_map[0, pixel_inds] += 6e-1

In [ ]:
# Convert the vector in theta phi to lon lat
hp.gnomview(output_map[0], rot=[lon.value, lat.value], unit="microK", xsize=200, nest=True)
hp.graticule()

In [ ]:
output_file_with_source = os.path.join(out_dir, f"{res}_cl_map_{int((source_size.to(u.arcmin)).value)}arcmin.fits")
hp.write_map(output_file_with_source, output_map, nest=True, overwrite=True)

# Make Area Fits File

In [4]:
import numpy as np
import astropy.units as u
from pixell import enmap

# --- pulled straight from blank_map.ipynb ---
center_ra_deg = lon.to_value(u.deg)     # 290.0
center_dec_deg = lat.to_value(u.deg)    # -60.0

patch_width_deg = 10.0   # scheduled patch width
fov_deg = 1.3            # focal-plane FOV
padding_factor = 1.2

half_extent_deg = 0.5 * (patch_width_deg + fov_deg) * padding_factor
ra_min, ra_max = center_ra_deg - half_extent_deg, center_ra_deg + half_extent_deg
dec_min, dec_max = center_dec_deg - half_extent_deg, center_dec_deg + half_extent_deg

box = np.radians([[dec_min, ra_min], [dec_max, ra_max]])
area_res = np.radians(2.0 / 60.0)  # 2 arcmin

shape, wcs = enmap.geometry(box, res=area_res, proj="car")
enmap.write_map_geometry("area.fits", shape, wcs)
print(f"shape={shape}, RA=[{ra_min:.2f},{ra_max:.2f}], Dec=[{dec_min:.2f},{dec_max:.2f}]")

shape=(np.int64(407), np.int64(407)), RA=[283.22,296.78], Dec=[-66.78,-53.22]
